In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn

In [40]:
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [41]:
def tokenize(text):
  text = text.lower()
  text = text.replace('?', '')
  text = text.replace("'", '')

  return text.split()

In [42]:
tokenize("What is the capital of India ?")

['what', 'is', 'the', 'capital', 'of', 'india']

In [43]:
vocab = {
    '<UNK>': 0
}

In [44]:
def build_vocab(row):
  tokenized_questions = tokenize(row['question'])
  tokenized_answers = tokenize(row['answer'])

  merged_token = tokenized_questions + tokenized_answers

  for token in merged_token:
    if token not in vocab:
      vocab[token] = len(vocab)

In [45]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [46]:
def text_to_indices(text, vocab):
  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [47]:
text_to_indices("What is the capital of India ?", vocab)

[1, 2, 3, 4, 5, 73]

In [48]:
import torch
from torch.utils.data import Dataset, DataLoader

In [49]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    num_questions = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    num_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(num_questions), torch.tensor(num_answer)

In [50]:
dataset = QADataset(df, vocab)

In [51]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [52]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_questions = self.embedding(question)

    rnn_outputs, final_hidden = self.rnn(embedded_questions)

    final_state = final_hidden[-1]

    output = self.fc(final_state)

    return output

In [53]:
learning_rate = 0.001
epochs = 100

In [54]:
model = SimpleRNN(len(vocab))

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [55]:
for epoch in range(epochs):
  total_loss = 0
  model.train()

  for question, answer in dataloader:
    optimizer.zero_grad()

    output = model(question)
    answer = answer.squeeze(0)

    loss = loss_fn(output, answer)
    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epochs {epoch+1} Loss {total_loss:4f}")

Epochs 1 Loss 524.107077
Epochs 2 Loss 456.659528
Epochs 3 Loss 379.449323
Epochs 4 Loss 312.804617
Epochs 5 Loss 260.339759
Epochs 6 Loss 212.086275
Epochs 7 Loss 167.325225
Epochs 8 Loss 129.646612
Epochs 9 Loss 99.511232
Epochs 10 Loss 75.639329
Epochs 11 Loss 57.824750
Epochs 12 Loss 45.138049
Epochs 13 Loss 35.784162
Epochs 14 Loss 29.124783
Epochs 15 Loss 23.893810
Epochs 16 Loss 19.917651
Epochs 17 Loss 16.789531
Epochs 18 Loss 14.295703
Epochs 19 Loss 12.205616
Epochs 20 Loss 10.521038
Epochs 21 Loss 9.201237
Epochs 22 Loss 8.116302
Epochs 23 Loss 7.167775
Epochs 24 Loss 6.409492
Epochs 25 Loss 5.737016
Epochs 26 Loss 5.183175
Epochs 27 Loss 4.686127
Epochs 28 Loss 4.259712
Epochs 29 Loss 3.906380
Epochs 30 Loss 3.573007
Epochs 31 Loss 3.284704
Epochs 32 Loss 3.026720
Epochs 33 Loss 2.792160
Epochs 34 Loss 2.584931
Epochs 35 Loss 2.402058
Epochs 36 Loss 2.235090
Epochs 37 Loss 2.076330
Epochs 38 Loss 1.937750
Epochs 39 Loss 1.809063
Epochs 40 Loss 1.692295
Epochs 41 Loss 1.5856

In [56]:
vocab_index_to_word = {index: word for word, index in vocab.items()}

In [57]:
def predict(model, question_tensor, vocab_index_to_word):
    model.eval()

    with torch.no_grad():
        output = model(question_tensor)

        predicted_index = torch.argmax(output, dim=1).item()

    return vocab_index_to_word[predicted_index]

In [63]:
test_question = "What is the capital of India ?"

indexed_question = text_to_indices(test_question, vocab)

question_tensor = torch.tensor(indexed_question, dtype=torch.long).unsqueeze(0)

predicted_word = predict(model, question_tensor, vocab_index_to_word)
print(f"Question: {test_question}")
print(f"Predicted Answer: {predicted_word}")

Question: What is the capital of India ?
Predicted Answer: delhi
